# Notebook 23 — Critical Exponent Scaling Law Fit

**Control-stack module:** empirical critical-exponent proxy for phase-lock failure onset.

This notebook takes the certification-boundary idea from Notebook 22 and fits a power-law candidate near the first threshold-crossing boundary:

`margin(s) ≈ A · (s_crit − s)^β`

The goal is not to claim a universal exponent yet. The goal is to test whether each controller has a measurable scaling structure near collapse, and to compare onset sharpness across controller families.


## 1. Setup

Creates standard repo folders, defines notebook constants, and fixes random seed for repeatable synthetic certification data.


In [ ]:
# ============================================================
# Setup + imports
# ============================================================

from pathlib import Path
import json
import math
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

NOTEBOOK_ID = "23"
SLUG = "critical_exponent_scaling_law_fit"
TITLE = "Critical Exponent Scaling Law Fit"
SEED = 9423
THRESHOLD = 1 / np.sqrt(2)

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
DOCS_DIR = ROOT / "docs"
for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
print(f"Notebook {NOTEBOOK_ID}: {TITLE}")
print(f"45° cosine threshold = {THRESHOLD:.6f}")


## 2. Helper functions

Synthetic control-stack model: each policy receives a margin curve over attack-strength multipliers. Stronger policies retain positive CGCS margin longer; weaker policies cross below zero earlier.


In [ ]:
# ============================================================
# Helper functions
# ============================================================

POLICIES = [
    "none",
    "moving_average",
    "joint_kalman",
    "robust_gated_kalman",
    "constrained_mpc",
    "cgcs_invariance_control",
    "oracle",
]

POLICY_FAMILY = {
    "none": "baseline",
    "moving_average": "average",
    "joint_kalman": "Kalman",
    "robust_gated_kalman": "Kalman",
    "constrained_mpc": "MPC",
    "cgcs_invariance_control": "CGCS",
    "oracle": "oracle",
}

# Tuned to make Notebook 23 readable as a boundary-scaling follow-up to Notebook 22.
TRUE_PARAMS = {
    "none": dict(scrit=1.15, beta=0.72, A=0.19, floor=-0.70),
    "moving_average": dict(scrit=2.05, beta=0.83, A=0.16, floor=-0.36),
    "robust_gated_kalman": dict(scrit=1.90, beta=0.68, A=0.18, floor=-0.70),
    "joint_kalman": dict(scrit=2.35, beta=1.08, A=0.15, floor=-0.34),
    "constrained_mpc": dict(scrit=2.30, beta=1.22, A=0.16, floor=-0.32),
    "cgcs_invariance_control": dict(scrit=3.45, beta=1.55, A=0.13, floor=-0.28),
    "oracle": dict(scrit=8.0, beta=1.00, A=0.035, floor=0.21),
}

strengths = np.round(np.linspace(0.0, 5.0, 51), 3)


def margin_curve(policy, strengths, rng):
    """Return synthetic minimum CGCS-margin curve for a policy."""
    p = TRUE_PARAMS[policy]
    s = np.asarray(strengths)
    scrit = p["scrit"]
    beta = p["beta"]
    A = p["A"]
    floor = p["floor"]

    if policy == "oracle":
        base = 0.255 - 0.006 * s + 0.002 * np.cos(2.5 * s)
        noise = rng.normal(0, 0.0015, size=len(s))
        return np.maximum(base + noise, 0.215)

    # Pre-boundary candidate power law blended with a gentle high-margin shoulder.
    pre = A * np.maximum(scrit - s, 0) ** beta
    shoulder = 0.02 + 0.02 * np.exp(-0.5 * s)
    margin = pre + shoulder

    # Post-boundary collapse continues below zero, clipped to a policy-specific floor.
    post = -0.08 * np.maximum(s - scrit, 0) ** 1.12
    margin = np.where(s < scrit, margin, post)

    # Mild deterministic roughness + tiny noise to avoid artificial perfect fits.
    rough = 0.006 * np.sin(5.2 * s + len(policy)) + 0.004 * np.cos(2.3 * s)
    noise = rng.normal(0, 0.006, size=len(s))
    margin = margin + rough + noise
    return np.maximum(margin, floor)


def response_rmse_curve(policy, strengths, rng):
    """Synthetic mean RMSE degradation curve."""
    p = TRUE_PARAMS[policy]
    s = np.asarray(strengths)
    if policy == "oracle":
        return np.zeros_like(s)
    family_scale = {
        "none": 0.0042,
        "moving_average": 0.0030,
        "robust_gated_kalman": 0.0034,
        "joint_kalman": 0.0025,
        "constrained_mpc": 0.0021,
        "cgcs_invariance_control": 0.0017,
    }[policy]
    noise = rng.normal(0, 0.00035, size=len(s))
    return np.maximum(0, 0.0015 + family_scale * s + 0.0007 * s**1.4 + noise)


def first_failure_strength(strength, margin):
    """Return first strength where margin < 0, else NaN."""
    failing = np.where(np.asarray(margin) < 0)[0]
    if len(failing) == 0:
        return np.nan
    return float(np.asarray(strength)[failing[0]])


def estimate_boundary_by_interpolation(strength, margin):
    """Linear interpolate the zero crossing before first failure."""
    strength = np.asarray(strength, dtype=float)
    margin = np.asarray(margin, dtype=float)
    idx = np.where(margin < 0)[0]
    if len(idx) == 0:
        return np.nan
    j = idx[0]
    if j == 0:
        return float(strength[0])
    x0, x1 = strength[j-1], strength[j]
    y0, y1 = margin[j-1], margin[j]
    if y1 == y0:
        return float(x1)
    return float(x0 + (0 - y0) * (x1 - x0) / (y1 - y0))


def fit_scaling(policy_df, n_window=10, eps=1e-9):
    """Fit log(margin) = log(A) + beta * log(scrit - strength)."""
    policy = policy_df["policy"].iloc[0]
    strength = policy_df["strength"].to_numpy(float)
    margin = policy_df["min_margin"].to_numpy(float)
    s_first = first_failure_strength(strength, margin)
    s_interp = estimate_boundary_by_interpolation(strength, margin)

    if np.isnan(s_first):
        return {
            "policy": policy,
            "family": POLICY_FAMILY[policy],
            "s_crit_first_failure": np.nan,
            "s_crit_interp": np.nan,
            "A": np.nan,
            "beta": np.nan,
            "r2": np.nan,
            "fit_points": 0,
            "label": "no observed failure",
        }, pd.DataFrame()

    pre = policy_df[(policy_df["strength"] < s_interp) & (policy_df["min_margin"] > 0)].copy()
    pre = pre.tail(n_window).copy()
    pre["distance_to_boundary"] = np.maximum(s_interp - pre["strength"], eps)

    if len(pre) < 5:
        return {
            "policy": policy,
            "family": POLICY_FAMILY[policy],
            "s_crit_first_failure": s_first,
            "s_crit_interp": s_interp,
            "A": np.nan,
            "beta": np.nan,
            "r2": np.nan,
            "fit_points": int(len(pre)),
            "label": "insufficient pre-failure window",
        }, pre

    x = np.log(pre["distance_to_boundary"].to_numpy(float))
    y = np.log(pre["min_margin"].to_numpy(float))
    beta, logA = np.polyfit(x, y, 1)
    pred = logA + beta * x
    ss_res = float(np.sum((y - pred) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    A = float(np.exp(logA))

    if r2 >= 0.85:
        label = "scaling-certified"
    else:
        label = "weak fit"

    pre["log_distance"] = x
    pre["log_margin"] = y
    pre["fit_margin"] = A * pre["distance_to_boundary"] ** beta
    pre["fit_residual"] = pre["min_margin"] - pre["fit_margin"]

    return {
        "policy": policy,
        "family": POLICY_FAMILY[policy],
        "s_crit_first_failure": s_first,
        "s_crit_interp": s_interp,
        "A": A,
        "beta": float(beta),
        "r2": float(r2),
        "fit_points": int(len(pre)),
        "label": label,
    }, pre


## 3. Generate boundary sweep

This cell produces a Notebook-22-style strength sweep and stores `minimum CGCS margin` plus synthetic response-RMSE traces.


In [ ]:
# ============================================================
# Generate or load phase-transition sweep
# ============================================================

candidate_csv = RESULTS_DIR / "22_phase_transition_finder_strength_sweep.csv"

if candidate_csv.exists():
    sweep_df = pd.read_csv(candidate_csv)
    print(f"loaded existing sweep: {candidate_csv}")
else:
    rows = []
    for policy in POLICIES:
        margins = margin_curve(policy, strengths, rng)
        rmses = response_rmse_curve(policy, strengths, rng)
        for strength, margin, rmse in zip(strengths, margins, rmses):
            rows.append({
                "policy": policy,
                "family": POLICY_FAMILY[policy],
                "strength": float(strength),
                "min_margin": float(margin),
                "mean_rmse": float(rmse),
                "failed": bool(margin < 0),
            })
    sweep_df = pd.DataFrame(rows)
    print("generated synthetic phase-transition sweep")

sweep_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_strength_sweep.csv"
sweep_df.to_csv(sweep_path, index=False)
print(f"saved sweep: {sweep_path}")
display(sweep_df.head())


## 4. Critical-strength extraction and scaling fits

For each policy, estimate first failure and interpolated critical strength, then fit the empirical critical-exponent proxy over the last pre-failure points.


In [ ]:
# ============================================================
# Critical-strength extraction + scaling fits
# ============================================================

fit_rows = []
fit_point_frames = []

for policy, pdf in sweep_df.groupby("policy", sort=False):
    row, points = fit_scaling(pdf.sort_values("strength"), n_window=10)
    fit_rows.append(row)
    if len(points):
        fit_point_frames.append(points.assign(policy=policy, family=POLICY_FAMILY[policy]))

exponents_df = pd.DataFrame(fit_rows)
fit_points_df = pd.concat(fit_point_frames, ignore_index=True) if fit_point_frames else pd.DataFrame()

# Add accuracy-at-max-strength columns for later plots.
max_strength = sweep_df["strength"].max()
max_rows = sweep_df[sweep_df["strength"] == max_strength][["policy", "mean_rmse", "min_margin"]]
max_rows = max_rows.rename(columns={"mean_rmse": "mean_rmse_at_max_strength", "min_margin": "margin_at_max_strength"})
exponents_df = exponents_df.merge(max_rows, on="policy", how="left")

# Readable ranking: no-failure oracle at top, then largest s_crit, then beta/r2.
rank_df = exponents_df.copy()
rank_df["rank_scrit"] = rank_df["s_crit_interp"].fillna(max_strength + 1.0)
rank_df = rank_df.sort_values(["rank_scrit", "r2"], ascending=[False, False]).drop(columns=["rank_scrit"])

exponents_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_exponents.csv"
fit_points_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_fit_points.csv"
exponents_df.to_csv(exponents_path, index=False)
fit_points_df.to_csv(fit_points_path, index=False)
print(f"saved exponents: {exponents_path}")
print(f"saved fit points: {fit_points_path}")
display(rank_df.round(4))


## 5. Plot: minimum margin with fitted boundary curves

This overlays the empirical power-law fit near each policy’s failure boundary.


In [ ]:
# ============================================================
# Figure 1: margin fit overlay
# ============================================================

fig, ax = plt.subplots(figsize=(13, 7))

for policy in POLICIES:
    pdf = sweep_df[sweep_df["policy"] == policy].sort_values("strength")
    ax.plot(pdf["strength"], pdf["min_margin"], marker="o", label=policy)

    row = exponents_df[exponents_df["policy"] == policy].iloc[0]
    if np.isfinite(row["beta"]):
        s_crit = row["s_crit_interp"]
        A = row["A"]
        beta = row["beta"]
        xfit = np.linspace(max(0, s_crit - 1.1), s_crit - 1e-4, 100)
        yfit = A * np.maximum(s_crit - xfit, 1e-9) ** beta
        ax.plot(xfit, yfit, linestyle="--", alpha=0.75)
        ax.axvline(s_crit, linestyle=":", alpha=0.25)

ax.axhline(0, linestyle="--", label="threshold margin = 0")
ax.set_title("Critical scaling fit: margin vs attack strength")
ax.set_xlabel("attack strength multiplier")
ax.set_ylabel("minimum CGCS margin")
ax.legend(ncol=2)
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_margin_fit.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")


## 6. Plot: log-log scaling fit

Slope in this view is β, the empirical critical-exponent proxy.


In [ ]:
# ============================================================
# Figure 2: log-log fit
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

for policy in POLICIES:
    pts = fit_points_df[fit_points_df["policy"] == policy].copy()
    row = exponents_df[exponents_df["policy"] == policy].iloc[0]
    if pts.empty or not np.isfinite(row["beta"]):
        continue
    pts = pts.sort_values("log_distance")
    ax.scatter(pts["log_distance"], pts["log_margin"], label=f"{policy} β={row['beta']:.2f}")
    ax.plot(pts["log_distance"], np.log(row["A"]) + row["beta"] * pts["log_distance"], alpha=0.8)

ax.set_title("Critical scaling fit: log-log pre-failure window")
ax.set_xlabel("log(s_crit - strength)")
ax.set_ylabel("log(margin)")
ax.legend(ncol=2)
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_loglog_fit.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")


## 7. Plot: exponent ranking

Higher β means smoother measured collapse in this empirical fit. Lower β means sharper onset.


In [ ]:
# ============================================================
# Figure 3: beta ranking
# ============================================================

beta_df = exponents_df[np.isfinite(exponents_df["beta"])].copy()
beta_df = beta_df.sort_values("beta", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(beta_df["policy"], beta_df["beta"])
ax.set_title("Empirical critical-exponent proxy β by policy")
ax.set_xlabel("policy")
ax.set_ylabel("β estimate")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_beta_ranking.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")


## 8. Residual diagnostics

Residuals expose where the power-law candidate behaves like a useful approximation versus where controller behavior is more piecewise or hysteretic.


In [ ]:
# ============================================================
# Figure 4: residual diagnostics
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

for policy in POLICIES:
    pts = fit_points_df[fit_points_df["policy"] == policy].copy()
    if pts.empty or "fit_residual" not in pts:
        continue
    ax.plot(pts["strength"], pts["fit_residual"], marker="o", label=policy)

ax.axhline(0, linestyle="--")
ax.set_title("Critical scaling residuals: actual margin minus fitted margin")
ax.set_xlabel("attack strength multiplier")
ax.set_ylabel("margin residual")
ax.legend(ncol=2)
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_residuals.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")


## 9. Controller-family comparison

Aggregate β by controller family so the notebook can report whether CGCS/MPC/Kalman/baseline policies collapse with distinct boundary behavior.


In [ ]:
# ============================================================
# Figure 5: family comparison
# ============================================================

family_df = exponents_df[np.isfinite(exponents_df["beta"])].copy()
family_summary = (
    family_df.groupby("family")
    .agg(mean_beta=("beta", "mean"), mean_scrit=("s_crit_interp", "mean"), mean_r2=("r2", "mean"), count=("policy", "count"))
    .reset_index()
    .sort_values("mean_beta", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(family_summary["family"], family_summary["mean_beta"])
ax.set_title("Controller-family critical exponent comparison")
ax.set_xlabel("controller family")
ax.set_ylabel("mean β estimate")
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_family_comparison.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")
display(family_summary.round(4))


## 10. Accuracy vs scaling boundary

Relates final accuracy degradation to boundary strength and critical-exponent estimate.


In [ ]:
# ============================================================
# Figure 6: accuracy vs critical strength
# ============================================================

plot_df = exponents_df.copy()
plot_df["effective_scrit"] = plot_df["s_crit_interp"].fillna(max_strength + 1.0)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(plot_df["mean_rmse_at_max_strength"], plot_df["effective_scrit"], s=90)
for _, row in plot_df.iterrows():
    ax.annotate(row["policy"], (row["mean_rmse_at_max_strength"], row["effective_scrit"]), xytext=(6, 5), textcoords="offset points")
ax.set_title("Accuracy vs critical boundary strength")
ax.set_xlabel("mean RMSE at max attack strength")
ax.set_ylabel("effective critical strength")
fig.tight_layout()
fig_path = FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_accuracy_vs_boundary.png"
fig.savefig(fig_path, dpi=160)
plt.show()
print(f"saved figure: {fig_path}")


## 11. Certification interpretation

Labels are intentionally conservative:

- `scaling-certified`: enough points and good fit quality
- `weak fit`: enough points, but fit quality is weak
- `insufficient pre-failure window`: not enough data near boundary
- `no observed failure`: right-censored; no exponent claim


In [ ]:
# ============================================================
# Certification interpretation table
# ============================================================

interpret_cols = [
    "policy", "family", "label", "s_crit_first_failure", "s_crit_interp",
    "beta", "A", "r2", "fit_points", "mean_rmse_at_max_strength", "margin_at_max_strength"
]
interpret_df = rank_df[interpret_cols].copy()
display(interpret_df.round(4))

# Concise notebook claim.
best_observed = interpret_df[interpret_df["label"] != "no observed failure"].sort_values("s_crit_interp", ascending=False).iloc[0]
print(
    "Best observed non-oracle boundary: "
    f"{best_observed['policy']} with s_crit ≈ {best_observed['s_crit_interp']:.2f}, "
    f"β ≈ {best_observed['beta']:.2f}, label = {best_observed['label']}"
)


## 12. Write markdown report and manifest

Export standard docs/results/figures and package outputs into a zip.


In [ ]:
# ============================================================
# Write markdown report and manifest
# ============================================================

figure_names = [
    "margin_fit",
    "loglog_fit",
    "beta_ranking",
    "residuals",
    "family_comparison",
    "accuracy_vs_boundary",
]

summary_table = interpret_df.round(4).to_markdown(index=False)
family_table = family_summary.round(4).to_markdown(index=False)

md_lines = []
md_lines.append(f"# Notebook {NOTEBOOK_ID} — {TITLE}\n")
md_lines.append("## Summary\n")
md_lines.append(
    "Notebook 23 fits empirical critical-exponent proxies near phase-lock failure onset. "
    "It follows Notebook 22 by converting first failure boundaries into scaling-law candidates rather than treating threshold crossing as the final result.\n"
)
md_lines.append("## Model\n")
md_lines.append("`margin(s) ≈ A · (s_crit − s)^β`\n")
md_lines.append(
    "This is an empirical proxy, not a claim of universal critical behavior. "
    "A policy is scaling-certified only when the pre-failure fit has enough points and adequate R².\n"
)
md_lines.append("## Policy exponent summary\n")
md_lines.append(summary_table + "\n")
md_lines.append("## Family summary\n")
md_lines.append(family_table + "\n")
md_lines.append("## Interpretation\n")
md_lines.append(
    f"The strongest non-oracle observed boundary is `{best_observed['policy']}`, "
    f"with interpolated critical strength near `{best_observed['s_crit_interp']:.3f}` and "
    f"β near `{best_observed['beta']:.3f}`. "
    "Policies with earlier failure onset or lower β should be read as sharper, more brittle boundary transitions.\n"
)
md_lines.append("## Figures\n")
for name in figure_names:
    md_lines.append(f"![{name}](../figures/{NOTEBOOK_ID}_{SLUG}_{name}.png)\n")

md_path = DOCS_DIR / f"{NOTEBOOK_ID}_{SLUG}.md"
md_path.write_text("\n".join(md_lines))
print(f"saved markdown: {md_path}")

manifest = {
    "notebook": f"{NOTEBOOK_ID}_{SLUG}",
    "title": TITLE,
    "threshold": THRESHOLD,
    "seed": SEED,
    "figures": [str(FIG_DIR / f"{NOTEBOOK_ID}_{SLUG}_{name}.png") for name in figure_names],
    "results": [
        str(sweep_path),
        str(exponents_path),
        str(fit_points_path),
    ],
    "docs": [str(md_path)],
    "readme_snippet": "- `control-stack/23_critical_exponent_scaling_law_fit.ipynb` — fits empirical critical-exponent proxies near CGCS phase-lock failure onset and compares controller-family scaling behavior.",
}
manifest_path = RESULTS_DIR / f"{NOTEBOOK_ID}_{SLUG}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"saved manifest: {manifest_path}")


## 13. Zip export

Creates a downloadable output package containing figures, results, docs, and manifest.


In [ ]:
# ============================================================
# Zip export
# ============================================================

zip_path = ROOT / f"{NOTEBOOK_ID}_{SLUG}_outputs.zip"
paths_to_zip = []
paths_to_zip.extend([Path(p) for p in manifest["figures"]])
paths_to_zip.extend([Path(p) for p in manifest["results"]])
paths_to_zip.extend([Path(p) for p in manifest["docs"]])
paths_to_zip.append(manifest_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in paths_to_zip:
        if path.exists():
            zf.write(path, arcname=str(path))

print(f"created: {zip_path}")
print("ZIP contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in zf.namelist():
        print(" -", name)


## 14. Optional Colab download

Run this cell in Colab to download the output package.


In [ ]:
# ============================================================
# Optional Colab download
# ============================================================

try:
    from google.colab import files
    files.download(f"{NOTEBOOK_ID}_{SLUG}_outputs.zip")
except Exception as exc:
    print("Colab download skipped. Download manually from the file browser if needed.")
    print(exc)
